# NEWSBlogTrafficTest in Google Colab

Run full project features (Dashboard, Threaded Journey, Locust, Browser Matrix) directly in Colab.


In [ ]:
import pathlib
import subprocess

REPO_URL = "https://github.com/psainaveen12/NewsBlogTrafficGen.git"
PROJECT_ROOT = pathlib.Path("/content/NEWSBlogTrafficTest")

if not PROJECT_ROOT.exists():
    if not REPO_URL.strip():
        raise ValueError("Set REPO_URL before running this cell.")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
%cd /content/NEWSBlogTrafficTest


In [ ]:
!bash NewsBlogTraffic_Colab/bootstrap_colab.sh /content/NEWSBlogTrafficTest


In [ ]:
!python3 scripts/validate_config.py --config config/sites.yaml


## Launch Dashboard (Streamlit)


In [ ]:
import atexit
import subprocess
import time
from google.colab import output

dashboard_proc = subprocess.Popen(
    ["python3", "scripts/run_dashboard.py", "--host", "0.0.0.0", "--port", "8501", "--headless"],
    cwd="/content/NEWSBlogTrafficTest",
)

def _cleanup_dashboard():
    if dashboard_proc.poll() is None:
        dashboard_proc.terminate()

atexit.register(_cleanup_dashboard)
time.sleep(5)
print("Dashboard PID:", dashboard_proc.pid)
output.serve_kernel_port_as_iframe(8501, height=1000)


In [ ]:
# Stop dashboard when needed
if 'dashboard_proc' in globals() and dashboard_proc.poll() is None:
    dashboard_proc.terminate()
    print('Dashboard stopped')
else:
    print('Dashboard is not running')


## Run Runners from Colab


In [ ]:
# Threaded Journey (example)
!python3 scripts/threaded_browser_journey.py --config config/sites.yaml --threads 5 --browser chromium --min-clicks 3 --max-clicks 4 --min-page-browse-seconds 20 --max-page-browse-seconds 30 --min-scroll-seconds 6 --max-scroll-seconds 12 --min-scroll-pause-seconds 1 --max-scroll-pause-seconds 3 --min-post-scroll-click-delay-seconds 2 --max-post-scroll-click-delay-seconds 5


In [ ]:
# Locust Headless Load (example)
!python3 scripts/run_locust.py --config config/sites.yaml --users 100 --spawn-rate 10 --run-time 10m --results-dir results


In [ ]:
# Browser Matrix (example)
!python3 scripts/browser_matrix.py --config config/sites.yaml --paths-per-target 2 --browsers chromium firefox webkit --output-json results/browser-matrix.json


In [ ]:
# Optional lightweight smoke run
!python3 NewsBlogTraffic_Colab/run_all_smoke.py --project-root /content/NEWSBlogTrafficTest --config config/sites.yaml
